In [0]:
import os
import data_ingestion
from pyspark.sql.functions import current_timestamp

# --- BRONZE LAYER: Professional Ingestion ---

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
if not API_KEY:
    raise ValueError("ALPHA_VANTAGE_API_KEY environment variable is not set. Please check your .env file.")

# 1. Get data, convert to Spark DataFrame
pdf = data_ingestion.fetch_bank_data(API_KEY)
bronze_df = spark.createDataFrame(pdf)

# 2. Add Metadata
bronze_final = bronze_df.withColumn("ingestion_timestamp", current_timestamp())

# 3. Write to Delta Table
bronze_table = "main.market_project.bronze_bank_prices"
bronze_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(bronze_table)

print(f"Success: Bronze table '{bronze_table}' created with ingestion timestamp.")